
# Point Cloud Global Registration Test Bench

This notebook:

1. Finds a reference point cloud named `main.txt` and all variations named `main_var_*.txt` inside a dataset folder.
2. Parses your custom text format, including metadata comments like `timestamp` and `transform_matrix`.
3. Applies **random translation + random yaw rotation** to each variation while keeping the chosen **up axis fixed**.
4. Runs **global registration** using the Open3D pipeline from the official tutorial:
   - voxel downsampling
   - normal estimation
   - FPFH feature extraction
   - RANSAC feature-based global registration
   - ICP refinement
5. Measures:
   - **registration quality** using Open3D metrics (`fitness`, `inlier_rmse`)
   - **pose recovery error** against the known synthetic perturbation **only when the original variation is already aligned to the reference pose**
6. Creates **partial scans** from the variations and repeats the same experiment.

## Expected file layout

```text
your_scene/
├── main.txt
├── main_empty.txt
├── main_var_1.txt
├── main_var_2.txt
├── main_var_3.txt
└── ...
```



In [ ]:
import copy
import math
import re
import sys
import time
from pathlib import Path
sys.path.insert(0, '../')
import drm
from drm import align
import trimesh

import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
import pandas as pd
from tqdm.auto import tqdm

%load_ext autoreload
%autoreload 2

In [ ]:
# ----------------------------
# Configuration
# ----------------------------
DATASET_DIR = Path(r"../../../datasets/V-Scan/data")
REFERENCE_NAME = "main.txt"
EMPTY_SCENE_NAME = "main_empty.txt"
VARIATION_GLOB = "main_var_*.txt"

UP_AXIS = "z"                 # one of: "x", "y", "z"
VOXEL_SIZE = 0.2             # tune for your dataset scale
USE_FAST_GLOBAL = False       # False = tutorial-style RANSAC, True = fast global registration
RUN_ICP_REFINEMENT = True

# Random perturbation settings
SEED = 10
NUM_TRIALS_PER_VARIATION = 3
MAX_TRANSLATION = 3
MAX_YAW_DEG = 90.0

# Partial scan settings
RUN_PARTIAL_SCAN_EXPERIMENT = True
PARTIAL_KEEP_FRACTIONS = [0.8, 0.6, 0.4]
PARTIAL_MODE = "crop"         # "crop" or "random_points"

# Visualization
SHOW_SAMPLE_ALIGNMENT = False
SHOW_PARTIAL_EXAMPLES = False


## Evaluation functions

These function can evaluate subfolders or whole datasets

In [ ]:
def evaluate_variations_alignment(
    ref_pcd,
    var_pcds,
    empty_pcd,
    translation_bounds=((-0.5, 0.5), (-0.1, 0.1), (-0.5, 0.5)),
    rotation_bounds_deg=(-20.0, 20.0),
    voxel_size=VOXEL_SIZE,
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
    seed=None,
):
    """
    For each variation pointcloud:
      - use its scanner pose (var_poses[i]) as crop origin
      - create one or more random partial crops
      - randomly transform each crop
      - register moved crop to reference scene
      - register moved crop to empty scene
      - report GT transform, estimated transforms, and pose errors

    Requires:
      - drm.radial_fov_crop_pointcloud(...)
      - drm.randomly_transform_pointcloud(...)
      - register_pair(...)
    """
    import numpy as np

    rng = np.random.default_rng(seed)
    results = []

    def rotation_error_deg(T_est, T_gt):
        R_err = T_est[:3, :3] @ T_gt[:3, :3].T
        trace_val = np.clip((np.trace(R_err) - 1.0) / 2.0, -1.0, 1.0)
        return float(np.degrees(np.arccos(trace_val)))

    for var_idx, var_pcd in enumerate(var_pcds):
        print(f"Processing variation {var_idx+1}/{len(var_pcds)}")
        move_seed = int(rng.integers(0, 1_000_000_000))

        # 2) random perturbation
        moved_pcd, applied_transform = drm.randomly_transform_pointcloud(
            var_pcd,
            translation_bounds=translation_bounds,
            rotate=True,
            up_axis="z",  # change if needed
            rotation_bounds_deg=rotation_bounds_deg,
            seed=move_seed,
        )

        gt_transform = np.linalg.inv(applied_transform)

        # 3) align to reference
        ref_reg = drm.align.register_pair(
            source=moved_pcd,
            target=ref_pcd,
            voxel_size=voxel_size,
            use_fast_global=use_fast_global,
            run_icp_refinement=run_icp_refinement,
        )

        ref_est = ref_reg["final_transform"]
        ref_err = ref_est @ np.linalg.inv(gt_transform)

        # 4) align to empty
        empty_reg = drm.align.register_pair(
            source=moved_pcd,
            target=empty_pcd,
            voxel_size=voxel_size,
            use_fast_global=use_fast_global,
            run_icp_refinement=run_icp_refinement,
        )

        empty_est = empty_reg["final_transform"]
        empty_err = empty_est @ np.linalg.inv(gt_transform)

        results.append({
            "variation_idx": var_idx,
            "status": "ok",
            "crop_idx": 0,

            "moved_pcd": moved_pcd,

            "ground_truth_transform": gt_transform,

            # reference metrics
            "reference_estimated_transform": ref_est,
            "reference_translation_error": float(np.linalg.norm(ref_err[:3, 3])),
            "reference_rotation_error_deg": rotation_error_deg(ref_est, gt_transform),
            "reference_final_fitness": ref_reg["final_fitness"],
            "reference_final_rmse": ref_reg["final_rmse"],

            # empty metrics
            "empty_estimated_transform": empty_est,
            "empty_translation_error": float(np.linalg.norm(empty_err[:3, 3])),
            "empty_rotation_error_deg": rotation_error_deg(empty_est, gt_transform),
            "empty_final_fitness": empty_reg["final_fitness"],
            "empty_final_rmse": empty_reg["final_rmse"],
        })

    return results


def evaluate_partial_variations_alignment(
    ref_pcd,
    var_pcds,
    var_poses,
    empty_pcd,
    num_random_crops_per_variation=1,
    horizontal_fov_deg=90.0,
    vertical_fov_deg=45.0,
    translation_bounds=((-0.5, 0.5), (-0.1, 0.1), (-0.5, 0.5)),
    rotation_bounds_deg=(-20.0, 20.0),
    voxel_size=VOXEL_SIZE,
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
    seed=None,
):
    """
    For each variation pointcloud:
      - use its scanner pose (var_poses[i]) as crop origin
      - create one or more random partial crops
      - randomly transform each crop
      - register moved crop to reference scene
      - register moved crop to empty scene
      - report GT transform, estimated transforms, and pose errors

    Requires:
      - drm.radial_fov_crop_pointcloud(...)
      - drm.randomly_transform_pointcloud(...)
      - register_pair(...)
    """
    import numpy as np

    rng = np.random.default_rng(seed)
    results = []

    def rotation_error_deg(T_est, T_gt):
        R_err = T_est[:3, :3] @ T_gt[:3, :3].T
        trace_val = np.clip((np.trace(R_err) - 1.0) / 2.0, -1.0, 1.0)
        return float(np.degrees(np.arccos(trace_val)))

    for var_idx, (var_pcd, crop_origin) in enumerate(zip(var_pcds, var_poses)):
        print(f"Processing variation {var_idx+1}/{len(var_pcds)}")
        crop_origin = np.asarray(crop_origin).astype(float)

        for crop_idx in range(num_random_crops_per_variation):
            print(f"  Crop {crop_idx+1}/{num_random_crops_per_variation}")
            crop_seed = int(rng.integers(0, 1_000_000_000))
            move_seed = int(rng.integers(0, 1_000_000_000))

            # 1) random crop using scanner pose as origin
            cropped_pcd, crop_mask, crop_info = drm.radial_fov_crop_pointcloud(
                var_pcd,
                horizontal_fov_deg=horizontal_fov_deg,
                vertical_fov_deg=vertical_fov_deg,
                horizontal_center_deg=None,
                vertical_center_deg=0.0,
                origin=crop_origin,
                seed=crop_seed,
            )

            if len(cropped_pcd.points) == 0:
                results.append({
                    "variation_idx": var_idx,
                    "crop_idx": crop_idx,
                    "status": "failed_empty_crop",
                    "crop_origin": crop_origin,
                    "crop_info": crop_info,
                })
                continue

            # 2) random perturbation
            moved_pcd, applied_transform = drm.randomly_transform_pointcloud(
                cropped_pcd,
                translation_bounds=translation_bounds,
                rotate=True,
                up_axis="z",  # change if needed
                rotation_bounds_deg=rotation_bounds_deg,
                seed=move_seed,
            )

            gt_transform = np.linalg.inv(applied_transform)

            # 3) align to reference
            ref_reg = drm.align.register_pair(
                source=moved_pcd,
                target=ref_pcd,
                voxel_size=voxel_size,
                use_fast_global=use_fast_global,
                run_icp_refinement=run_icp_refinement,
            )

            ref_est = ref_reg["final_transform"]
            ref_err = ref_est @ np.linalg.inv(gt_transform)

            # 4) align to empty
            empty_reg = drm.align.register_pair(
                source=moved_pcd,
                target=empty_pcd,
                voxel_size=voxel_size,
                use_fast_global=use_fast_global,
                run_icp_refinement=run_icp_refinement,
            )

            empty_est = empty_reg["final_transform"]
            empty_err = empty_est @ np.linalg.inv(gt_transform)

            results.append({
                "variation_idx": var_idx,
                "crop_idx": crop_idx,
                "status": "ok",
                "crop_origin": crop_origin,

                "cropped_pcd": cropped_pcd,
                "moved_pcd": moved_pcd,
                "crop_mask": crop_mask,
                "crop_info": crop_info,

                "ground_truth_transform": gt_transform,

                # reference metrics
                "reference_estimated_transform": ref_est,
                "reference_translation_error": float(np.linalg.norm(ref_err[:3, 3])),
                "reference_rotation_error_deg": rotation_error_deg(ref_est, gt_transform),
                "reference_final_fitness": ref_reg["final_fitness"],
                "reference_final_rmse": ref_reg["final_rmse"],

                # empty metrics
                "empty_estimated_transform": empty_est,
                "empty_translation_error": float(np.linalg.norm(empty_err[:3, 3])),
                "empty_rotation_error_deg": rotation_error_deg(empty_est, gt_transform),
                "empty_final_fitness": empty_reg["final_fitness"],
                "empty_final_rmse": empty_reg["final_rmse"],
            })

    return results

import numpy as np
import pandas as pd


def summarize_single_folder_by_variation(
    results,
    translation_error_thresh=0.10,
    rotation_error_thresh_deg=1.0,
):
    rows = []

    for r in results:
        if r.get("status") != "ok":
            continue

        row = {
            "variation_idx": r["variation_idx"],
            "crop_idx": r["crop_idx"],
        }

        for target in ["reference", "empty"]:
            dist = r.get(f"{target}_translation_error", np.nan)
            angle = r.get(f"{target}_rotation_error_deg", np.nan)
            rmse = r.get(f"{target}_final_rmse", np.nan)

            success = (
                np.isfinite(dist)
                and np.isfinite(angle)
                and dist <= translation_error_thresh
                and angle <= rotation_error_thresh_deg
            )

            row[f"{target}_success"] = success
            row[f"{target}_rmse"] = rmse
            row[f"{target}_distance_error"] = dist
            row[f"{target}_angle_error_deg"] = angle

        rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        return pd.DataFrame()

    summary_rows = []

    for variation_idx, g in df.groupby("variation_idx"):
        summary = {
            ("", "variation_idx"): variation_idx,
            ("", "num_trials"): len(g),
        }

        for target in ["reference", "empty"]:
            success_mask = g[f"{target}_success"].astype(bool)
            num_success = int(success_mask.sum())
            num_trials = len(g)

            summary[(target, "success_%")] = 100.0 * num_success / num_trials

            if num_success > 0:
                successful = g.loc[success_mask]

                summary[(target, "avg_rmse_success")] = successful[f"{target}_rmse"].mean()
                summary[(target, "avg_distance_error_success")] = successful[
                    f"{target}_distance_error"
                ].mean()
                summary[(target, "avg_angle_error_deg_success")] = successful[
                    f"{target}_angle_error_deg"
                ].mean()
            else:
                summary[(target, "avg_rmse_success")] = np.nan
                summary[(target, "avg_distance_error_success")] = np.nan
                summary[(target, "avg_angle_error_deg_success")] = np.nan

        summary_rows.append(summary)

    overview = pd.DataFrame(summary_rows)
    overview.columns = pd.MultiIndex.from_tuples(overview.columns)

    return overview.sort_values(("", "variation_idx")).reset_index(drop=True)

from pathlib import Path
import numpy as np
import pandas as pd


def evaluate_all_subfolders(
    main_folder,
    max_subfolders=None,
    reference_name=REFERENCE_NAME,
    empty_scene_name=EMPTY_SCENE_NAME,
    variation_glob=VARIATION_GLOB,
    voxel_size=VOXEL_SIZE,
    num_random_crops_per_variation=1,
    horizontal_fov_deg=90.0,
    vertical_fov_deg=45.0,
    translation_bounds=((-0.5, 0.5), (-0.1, 0.1), (-0.5, 0.5)),
    rotation_bounds_deg=(-20.0, 20.0),
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
    translation_error_thresh=0.10,
    rotation_error_thresh_deg=1.0,
    seed=None,
):
    """
    Evaluates every direct subfolder inside main_folder.

    Each subfolder should contain:
      - reference_name
      - empty_scene_name
      - files matching variation_glob

    Returns
    -------
    all_results : list[dict]
        All raw trial results, with folder info added.

    folder_overviews : dict[str, pd.DataFrame]
        One summary table per folder.

    combined_overview : pd.DataFrame
        All folder summaries combined into one table.
    """

    main_folder = Path(main_folder)
    rng = np.random.default_rng(seed)

    all_results = []
    folder_overviews = []
    overview_tables_by_folder = {}

    subfolders = sorted([p for p in main_folder.iterdir() if p.is_dir()])

    if max_subfolders is not None:
        subfolders = subfolders[:max_subfolders]

    for folder_idx, dataset_dir in enumerate(subfolders):
        print(f"\n=== Processing folder {folder_idx + 1}/{len(subfolders)}: {dataset_dir.name} ===")

        ref_path = dataset_dir / reference_name
        empty_scene_path = dataset_dir / empty_scene_name
        var_paths = sorted(dataset_dir.glob(variation_glob))

        if not ref_path.exists():
            print(f"Skipping {dataset_dir.name}: missing reference file {reference_name}")
            continue

        if not empty_scene_path.exists():
            print(f"Skipping {dataset_dir.name}: missing empty scene file {empty_scene_name}")
            continue

        if len(var_paths) == 0:
            print(f"Skipping {dataset_dir.name}: no variation files matching {variation_glob}")
            continue

        ref_pcd, _ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
        ref_pcd = ref_pcd.voxel_down_sample(voxel_size)

        empty_pcd, _ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
        empty_pcd = empty_pcd.voxel_down_sample(voxel_size)

        var_pcds = []
        var_poses = []

        for var_path in var_paths:
            var_pcd, _ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
            var_pcd = var_pcd.voxel_down_sample(voxel_size)

            var_pose = drm.read_transform_matrix(
                var_path,
                apply_unity_conversion=True
            )[:3, 3]

            var_pcds.append(var_pcd)
            var_poses.append(var_pose)

        folder_seed = int(rng.integers(0, 1_000_000_000))

        if(num_random_crops_per_variation < 1 or num_random_crops_per_variation is None):
            results = evaluate_variations_alignment(
                ref_pcd=ref_pcd,
                var_pcds=var_pcds,
                empty_pcd=empty_pcd,
                translation_bounds=translation_bounds,
                rotation_bounds_deg=rotation_bounds_deg,
                voxel_size=voxel_size,
                use_fast_global=use_fast_global,
                run_icp_refinement=run_icp_refinement,
                seed=folder_seed,
            )
        else:
            results = evaluate_partial_variations_alignment(
                ref_pcd=ref_pcd,
                var_pcds=var_pcds,
                var_poses=var_poses,
                empty_pcd=empty_pcd,
                num_random_crops_per_variation=num_random_crops_per_variation,
                horizontal_fov_deg=horizontal_fov_deg,
                vertical_fov_deg=vertical_fov_deg,
                translation_bounds=translation_bounds,
                rotation_bounds_deg=rotation_bounds_deg,
                voxel_size=voxel_size,
                use_fast_global=use_fast_global,
                run_icp_refinement=run_icp_refinement,
                seed=folder_seed,
            )

        for r in results:
            r["folder_idx"] = folder_idx
            r["folder_name"] = dataset_dir.name
            r["folder_path"] = str(dataset_dir)

        all_results.extend(results)

        overview = summarize_single_folder_by_variation(
            results,
            translation_error_thresh=translation_error_thresh,
            rotation_error_thresh_deg=rotation_error_thresh_deg,
        )

        if not overview.empty:
            overview.insert(0, ("", "folder_name"), dataset_dir.name)
            overview.insert(1, ("", "folder_idx"), folder_idx)

            overview_tables_by_folder[dataset_dir.name] = overview
            folder_overviews.append(overview)

    if len(folder_overviews) > 0:
        combined_raw = pd.concat(folder_overviews, ignore_index=True)

        metric_cols = [
            col for col in combined_raw.columns
            if col[1] not in ["folder_name", "folder_idx", "variation_idx"]
        ]

        grouped = combined_raw.groupby(("", "variation_idx"))

        combined_overview = grouped[metric_cols].mean().reset_index()

        combined_overview.insert(0, ("", "num_folders"), grouped.size().values)

        combined_overview = combined_overview.sort_values(
            ("", "variation_idx")
        ).reset_index(drop=True)

    else:
        combined_overview = pd.DataFrame()

    return all_results, overview_tables_by_folder, combined_overview


def transpose_variation_overview(df):
    """
    Transpose overview dataframe so:
      - variations become columns
      - metrics become rows

    Works with your MultiIndex overview tables.
    """

    import pandas as pd

    # Find variation column
    variation_col = ("", "variation_idx")

    if variation_col not in df.columns:
        raise ValueError("Could not find ('', 'variation_idx') column")

    # Use variation index as column labels
    temp = df.set_index(variation_col)

    # transpose
    out = temp.T

    # rename columns
    out.columns = [f"variation_{int(c)}" for c in out.columns]

    return out

## Single Folder Partial variations alignment Evaluation

- Have one reference pointcloud
- Have 3 variations
    - Each variation -> make a partial crop from the scanner origin
    - transform randomly
    - Calculate the alignment to the reference and the empty

In [ ]:
# Load the pointclouds from the folder
dataset_dir = Path(DATASET_DIR)
ref_path = dataset_dir / REFERENCE_NAME
empty_scene_path = dataset_dir / EMPTY_SCENE_NAME
var_paths = sorted(dataset_dir.glob(VARIATION_GLOB))
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_unity_conversion=True)
refPcd = refPcd.voxel_down_sample(VOXEL_SIZE)
emptyPcd,_ = drm.txt_pcd_to_open3d(empty_scene_path, apply_unity_conversion=True)
emptyPcd = emptyPcd.voxel_down_sample(VOXEL_SIZE)

varPcds = []
varPosses = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, apply_unity_conversion=True)
    varPcd = varPcd.voxel_down_sample(VOXEL_SIZE)
    posTransform = drm.read_transform_matrix(var_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
    varPcds.append(varPcd)
    varPosses.append(posTransform)

In [ ]:
# Optionally visualise them
newScene = drm.visualise_open3d_pointclouds([refPcd, emptyPcd] + varPcds)
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
results = evaluate_partial_variations_alignment(
    ref_pcd=refPcd,
    var_pcds=varPcds,
    var_poses=varPosses,
    empty_pcd=emptyPcd,
    num_random_crops_per_variation=3,
    horizontal_fov_deg=120,
    vertical_fov_deg=90,
    translation_bounds=((-1, 1), (-1, 1), (-0, 0)),
    rotation_bounds_deg=(-30, 30),
    voxel_size=0.1,
    use_fast_global=False,
    seed=SEED,
)

In [ ]:
overview = summarize_single_folder_by_variation(
    results,
    translation_error_thresh=0.10,
    rotation_error_thresh_deg=1.0,
)

display(overview)

## Full dataset

In [ ]:
all_results, folder_overviews, combined_overview = evaluate_all_subfolders(
    main_folder=DATASET_DIR,
    max_subfolders=50,
    num_random_crops_per_variation=NUM_TRIALS_PER_VARIATION,
    horizontal_fov_deg=120.0,
    vertical_fov_deg=90.0,
    translation_bounds=((-MAX_TRANSLATION, MAX_TRANSLATION), (-MAX_TRANSLATION, MAX_TRANSLATION), (-MAX_TRANSLATION, MAX_TRANSLATION)),
    rotation_bounds_deg=(-MAX_YAW_DEG, MAX_YAW_DEG),
    translation_error_thresh=0.20,
    rotation_error_thresh_deg=3.0,
    seed=SEED,
)

display(combined_overview)

In [ ]:
display(transpose_variation_overview(combined_overview))

In [ ]:
for folder_name, overview in folder_overviews.items():
    print(f"\n=== Overview for folder: {folder_name} ===")
    #display(overview)
    overview.to_csv(f"../_output/alignment_results_partial/{folder_name}_overview.csv", index=False)


In [ ]:
def export_overview_to_excel_and_latex(
    overview_df,
    excel_path="alignment_overview.xlsx",
    latex_path="alignment_overview.tex",
    sheet_name="overview",
    float_format="%.4f",
):
    """
    Exports your overview dataframe to:
      - Excel .xlsx
      - LaTeX .tex table

    Works with the transposed overview too.
    """
    from pathlib import Path

    excel_path = Path(excel_path)
    latex_path = Path(latex_path)

    overview_df.to_excel(excel_path, sheet_name=sheet_name)

    latex_str = overview_df.to_latex(
        index=True,
        escape=False,
        multicolumn=True,
        multirow=True,
        float_format=float_format,
    )

    latex_path.write_text(latex_str, encoding="utf-8")

    print(f"Saved Excel to: {excel_path}")
    print(f"Saved LaTeX to: {latex_path}")

    return excel_path, latex_path

In [ ]:
overview_t = transpose_variation_overview(combined_overview)

excel_path, latex_path = export_overview_to_excel_and_latex(
    overview_t,
    excel_path="combined_alignment_overview.xlsx",
    latex_path="combined_alignment_overview.tex",
)

## Step-By-Step

In [ ]:
# Dataset discovery + parsing
# ----------------------------
def discover_dataset(dataset_dir: Path, reference_name: str = REFERENCE_NAME, variation_glob: str = VARIATION_GLOB, empty_scene_name: str = EMPTY_SCENE_NAME):
    dataset_dir = Path(dataset_dir)
    reference_path = dataset_dir / reference_name
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference file not found: {reference_path}")

    empty_scene_path = dataset_dir / empty_scene_name
    if not empty_scene_path.exists():
        raise FileNotFoundError(f"Empty scene file not found: {empty_scene_path}")

    variation_paths = sorted(dataset_dir.glob(variation_glob))
    if not variation_paths:
        raise FileNotFoundError(f"No variation files matching '{variation_glob}' were found in {dataset_dir}")

    return reference_path, variation_paths, empty_scene_path


ref_path, var_paths, empty_scene_path = discover_dataset(DATASET_DIR)

In [ ]:
refPcd,_ = drm.txt_pcd_to_open3d(ref_path, apply_transform=False, apply_unity_conversion=True, inverse_transform=True)
refPcd_sub = refPcd.voxel_down_sample(VOXEL_SIZE)
posTransform = drm.read_transform_matrix(ref_path, apply_unity_conversion=True)[:3,3]  # just to check that the transform matrix is read correctly (it is)
#refPcd_sub.translate(-posTransform)


In [ ]:
varPcds = []
for var_path in var_paths:
    varPcd,_ = drm.txt_pcd_to_open3d(var_path, rotateX=True, mirrorY=True, apply_transform=True, apply_unity_conversion=True)
    varPcd_sub = varPcd.voxel_down_sample(VOXEL_SIZE)
    varPcds.append(varPcd_sub)

In [ ]:
newScene = drm.visualise_open3d_pointclouds([refPcd_sub])
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
partial_pcd, mask, info = drm.radial_fov_crop_pointcloud(
    refPcd_sub,
    horizontal_fov_deg=90,
    vertical_fov_deg=70,
    horizontal_center_deg=None,  # random
    vertical_center_deg=0.0,
    origin=posTransform,
)

print(info)

drm.visualise_open3d_pointclouds([partial_pcd]).show()

In [ ]:
movedPcd = drm.randomly_transform_pointcloud(
    partial_pcd,
    ((-1.0, 1.0), (-0.2, 0.2), (-1.0, 1.0)),
    rotate=True,
    up_axis="z",
    rotation_bounds_deg=(-180.0, 180.0),
    rotation_center=posTransform,
    seed=None,
)


In [ ]:
newScene = drm.visualise_open3d_pointclouds([partial_pcd, movedPcd[0]])
newScene.add_geometry(trimesh.creation.axis(origin_size=0.05, axis_length=1.0, axis_radius=0.005))
newScene.show()

In [ ]:
# 3) Register moved crop back to reference
reg_result = drm.align.register_pair(
    source=partial_pcd,
    target=movedPcd[0],
    voxel_size=0.1,
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
)

In [ ]:
reg_result["refined_result"].transformation

In [ ]:
drm.align.draw_registration_result(refPcd_sub, movedPcd[0], reg_result["refined_result"].transformation).show()

## Test Single partial same-scan registration

In [ ]:
def evaluate_single_random_partial_registration(
    reference_pcd,
    horizontal_fov_deg=90.0,
    vertical_fov_deg=45.0,
    crop_origin=(0.0, 0.0, 0.0),
    translation_bounds=((-3, 3), (-3, 3), (-3, 3)),
    rotation_bounds_deg=(-180, 180),
    voxel_size=0.2,
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
    seed=None,
):
    """
    Create one random partial crop from a reference point cloud, perturb it slightly,
    register it back to the reference, and report:
      - ground-truth transform
      - estimated transform
      - translation error
      - yaw error around Y-up axis

    Assumptions:
      - up axis is Y
      - randomly_transform_pointcloud(..., up_axis="y") rotates only around Y
      - source is the moved partial crop
      - target is the full reference point cloud
    """
    import numpy as np

    rng = np.random.default_rng(seed)
    crop_seed = int(rng.integers(0, 1_000_000_000))
    move_seed = int(rng.integers(0, 1_000_000_000))

    # 1) Create one random partial crop
    cropped_pcd, crop_mask, crop_info = drm.radial_fov_crop_pointcloud(
        reference_pcd,
        horizontal_fov_deg=horizontal_fov_deg,
        vertical_fov_deg=vertical_fov_deg,
        horizontal_center_deg=None,
        vertical_center_deg=0.0,
        origin=crop_origin,
        seed=crop_seed,
    )

    if len(cropped_pcd.points) == 0:
        return {
            "status": "failed_empty_crop",
            "crop_info": crop_info,
        }

    # 2) Apply a known small random perturbation
    moved_pcd, applied_transform = drm.randomly_transform_pointcloud(
        cropped_pcd,
        translation_bounds=translation_bounds,
        rotate=True,
        up_axis="z",
        rotation_bounds_deg=rotation_bounds_deg,
        seed=move_seed,
    )

    # 3) Register moved crop back to reference
    reg_result = drm.align.register_pair(
        source=moved_pcd,
        target=reference_pcd,
        voxel_size=voxel_size,
        use_fast_global=use_fast_global,
        run_icp_refinement=run_icp_refinement,
    )

    estimated_transform = reg_result["final_transform"]

    # 4) Ground truth registration transform = inverse of applied perturbation
    ground_truth_transform = np.linalg.inv(applied_transform)

    # 5) Errors
    transform_error = estimated_transform @ np.linalg.inv(ground_truth_transform)
    translation_error = float(np.linalg.norm(transform_error[:3, 3]))

    R_err = transform_error[:3, :3]
    rotation_trace = np.clip((np.trace(R_err) - 1.0) / 2.0, -1.0, 1.0)
    rotation_error_deg = float(np.degrees(np.arccos(rotation_trace)))

    return {
        "status": "ok",
        "cropped_pcd": cropped_pcd,
        "moved_pcd": moved_pcd,
        "crop_mask": crop_mask,
        "crop_info": crop_info,
        "ground_truth_transform": ground_truth_transform,
        "estimated_transform": estimated_transform,
        "transform_error": transform_error,
        "translation_error": translation_error,
        "rotation_error_deg": rotation_error_deg,
        **reg_result,
    }

In [ ]:
result = evaluate_single_random_partial_registration(refPcd_sub, translation_bounds=((-1.0, 1.0), (-1, 1), (-0.2, 0.2)), rotation_bounds_deg=(-30.0, 30.0), crop_origin=posTransform)

print("Status:", result["status"])
print("Ground truth transform:\n", result["ground_truth_transform"])
print("Estimated transform:\n", result["estimated_transform"])
print("Translation error:", result["translation_error"])
print("Rotation error (deg):", result["rotation_error_deg"])
print("Final fitness:", result["final_fitness"])
print("Final RMSE:", result["final_rmse"])

In [ ]:
drm.align.draw_registration_result(refPcd_sub, result["moved_pcd"]).show()

In [ ]:
drm.align.draw_registration_result(refPcd_sub, result["moved_pcd"], np.linalg.inv(result["estimated_transform"])).show()

In [ ]:
def benchmark_random_partial_registrations_from_reference(
    reference_pcd,
    num_trials=10,
    horizontal_fov_deg=90.0,
    vertical_fov_deg=45.0,
    crop_origin=(0.0, 0.0, 0.0),
    translation_bounds=((-1.0, 1.0), (-0.2, 0.2), (-1.0, 1.0)),
    rotate=True,
    rotation_bounds_deg=(-45.0, 45.0),
    voxel_size=VOXEL_SIZE,
    use_fast_global=USE_FAST_GLOBAL,
    run_icp_refinement=RUN_ICP_REFINEMENT,
    seed=None,
):
    """
    Create several random cropped pointclouds from a reference, randomly move them,
    and try to register them back to the reference using the existing pipeline
    functions.

    Required existing functions
    ---------------------------
    - radial_fov_crop_pointcloud_y_up(...)
    - randomly_transform_pointcloud(...)
    - register_pair(...)

    Returns
    -------
    results : list of dict
        One dict per trial with crop info, perturbation info, and registration output.
    """
    rng = np.random.default_rng(seed)
    results = []

    for trial_idx in range(num_trials):
        trial_seed_crop = int(rng.integers(0, 1_000_000_000))
        trial_seed_move = int(rng.integers(0, 1_000_000_000))

        cropped_pcd, crop_mask, crop_info = drm.radial_fov_crop_pointcloud(
            reference_pcd,
            horizontal_fov_deg=horizontal_fov_deg,
            vertical_fov_deg=vertical_fov_deg,
            horizontal_center_deg=None,
            vertical_center_deg=0.0,
            origin=crop_origin,
            seed=trial_seed_crop,
        )

        if len(cropped_pcd.points) == 0:
            results.append({
                "trial_idx": trial_idx,
                "status": "skipped_empty_crop",
                "crop_info": crop_info,
            })
            continue

        moved_pcd, applied_transform, transform_info = drm.randomly_transform_pointcloud(
            cropped_pcd,
            translation_bounds=translation_bounds,
            rotate=rotate,
            up_axis="y",
            rotation_bounds_deg=rotation_bounds_deg,
            rotate_around_center=True,
            seed=trial_seed_move,
        )

        reg_result = drm.align.register_pair(
            source=moved_pcd,
            target=reference_pcd,
            voxel_size=voxel_size,
            use_fast_global=use_fast_global,
            run_icp_refinement=run_icp_refinement,
        )

        results.append({
            "trial_idx": trial_idx,
            "status": "ok",
            "num_points_cropped": int(len(cropped_pcd.points)),
            "crop_info": crop_info,
            "crop_mask": crop_mask,
            "transform_info": transform_info,
            "applied_transform": applied_transform,
            **reg_result,
        })

    return results

In [ ]:

def draw_registration_result(source, target, transformation=np.eye(4)):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1.0, 0.706, 0.0])
    target_temp.paint_uniform_color([0.0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])


In [ ]:

summary_df = summarize_dataframe(
    results_df,
    group_cols=["experiment", "variation_name", "partial_keep_fraction"]
)
summary_df


In [ ]:

compact_cols = ["final_rmse", "final_fitness", "global_time_sec", "refine_time_sec"]
if ASSUME_VARIATIONS_ARE_PREALIGNED_FOR_POSE_ERROR:
    compact_cols = ["translation_error", "rotation_error_deg", "yaw_error_deg"] + compact_cols

compact_summary = (
    results_df.groupby(["experiment", "partial_keep_fraction"])[compact_cols]
    .mean()
    .reset_index()
    .sort_values(["experiment", "partial_keep_fraction"], ascending=[True, False])
)
compact_summary


In [ ]:

partial_df = results_df[results_df["experiment"] == "partial_scan"].copy()

if not partial_df.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    plot_df = (
        partial_df.groupby("partial_keep_fraction")[["final_rmse", "final_fitness"]]
        .mean()
        .reset_index()
        .sort_values("partial_keep_fraction")
    )
    ax.plot(plot_df["partial_keep_fraction"], plot_df["final_rmse"], marker="o", label="mean final_rmse")
    ax.plot(plot_df["partial_keep_fraction"], plot_df["final_fitness"], marker="o", label="mean final_fitness")
    ax.set_xlabel("Partial keep fraction")
    ax.set_ylabel("Metric value")
    ax.set_title("Registration quality vs partial scan size")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()

    if ASSUME_VARIATIONS_ARE_PREALIGNED_FOR_POSE_ERROR:
        fig, ax = plt.subplots(figsize=(8, 4))
        pose_plot_df = (
            partial_df.groupby("partial_keep_fraction")[["translation_error", "rotation_error_deg"]]
            .mean()
            .reset_index()
            .sort_values("partial_keep_fraction")
        )
        ax.plot(pose_plot_df["partial_keep_fraction"], pose_plot_df["translation_error"], marker="o", label="translation_error")
        ax.plot(pose_plot_df["partial_keep_fraction"], pose_plot_df["rotation_error_deg"], marker="o", label="rotation_error_deg")
        ax.set_xlabel("Partial keep fraction")
        ax.set_ylabel("Mean pose error")
        ax.set_title("Pose recovery error vs partial scan size")
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()
else:
    print("No partial scan results available.")


In [ ]:

output_dir = DATASET_DIR / "registration_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

results_csv = output_dir / "registration_results.csv"
summary_csv = output_dir / "registration_summary.csv"

results_df.to_csv(results_csv, index=False)
summary_df.to_csv(summary_csv)

print("Saved:")
print(" -", results_csv)
print(" -", summary_csv)



## Tuning tips

You will probably need to tune `VOXEL_SIZE` to match your dataset scale.

Typical starting points:

- dense small objects: `0.005` to `0.02`
- room-scale scans: `0.02` to `0.08`
- larger scenes: `0.08` to `0.20`

If the registration is unstable:

- try a larger `VOXEL_SIZE`
- retain more of the point cloud in partial scans
- switch `PARTIAL_MODE` between `"crop"` and `"random_points"`
- use `USE_FAST_GLOBAL = True` for speed, or `False` for tutorial-style RANSAC
- increase `NUM_TRIALS_PER_VARIATION`


In [ ]:
results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def summarize_partial_alignment_results(
    results,
    translation_success_threshold=0.10,
    rotation_success_threshold_deg=1.0,
    acceptable_translation_threshold=1.0,
    acceptable_rotation_threshold_deg=5.0,
):
    """
    Convert the returned results list into:
      - a clean pandas DataFrame
      - aggregate per-target / per-variation summaries
      - a few notebook-friendly plots

    Returns
    -------
    df : pd.DataFrame
        One row per trial.

    summary_target : pd.DataFrame
        Aggregated metrics for reference vs empty.

    summary_variation : pd.DataFrame
        Aggregated metrics by variation.

    figs : dict
        Dictionary of matplotlib figures.
    """

    rows = []
    for r in results:
        if r.get("status") != "ok":
            rows.append({
                "variation_idx": r.get("variation_idx"),
                "crop_idx": r.get("crop_idx"),
                "status": r.get("status"),
            })
            continue

        crop_info = r.get("crop_info", {})
        crop_origin = r.get("crop_origin", [np.nan, np.nan, np.nan])

        row = {
            "variation_idx": r.get("variation_idx"),
            "crop_idx": r.get("crop_idx"),
            "status": r.get("status"),

            "crop_origin_x": float(crop_origin[0]),
            "crop_origin_y": float(crop_origin[1]),
            "crop_origin_z": float(crop_origin[2]),

            "horizontal_center_deg": crop_info.get("horizontal_center_deg"),
            "vertical_center_deg": crop_info.get("vertical_center_deg"),
            "horizontal_fov_deg": crop_info.get("horizontal_fov_deg"),
            "vertical_fov_deg": crop_info.get("vertical_fov_deg"),
            "num_kept": crop_info.get("num_kept"),
            "num_total": crop_info.get("num_total"),
            "fraction_kept": crop_info.get("fraction_kept"),

            "reference_translation_error": r.get("reference_translation_error"),
            "reference_rotation_error_deg": r.get("reference_rotation_error_deg"),
            "reference_final_fitness": r.get("reference_final_fitness"),
            "reference_final_rmse": r.get("reference_final_rmse"),

            "empty_translation_error": r.get("empty_translation_error"),
            "empty_rotation_error_deg": r.get("empty_rotation_error_deg"),
            "empty_final_fitness": r.get("empty_final_fitness"),
            "empty_final_rmse": r.get("empty_final_rmse"),
        }

        rows.append(row)

    df = pd.DataFrame(rows)

    ok_mask = df["status"] == "ok"
    df_ok = df.loc[ok_mask].copy()

    # Success labels
    df_ok["reference_strict_success"] = (
        (df_ok["reference_translation_error"] <= translation_success_threshold) &
        (df_ok["reference_rotation_error_deg"] <= rotation_success_threshold_deg)
    )
    df_ok["empty_strict_success"] = (
        (df_ok["empty_translation_error"] <= translation_success_threshold) &
        (df_ok["empty_rotation_error_deg"] <= rotation_success_threshold_deg)
    )

    df_ok["reference_acceptable"] = (
        (df_ok["reference_translation_error"] <= acceptable_translation_threshold) &
        (df_ok["reference_rotation_error_deg"] <= acceptable_rotation_threshold_deg)
    )
    df_ok["empty_acceptable"] = (
        (df_ok["empty_translation_error"] <= acceptable_translation_threshold) &
        (df_ok["empty_rotation_error_deg"] <= acceptable_rotation_threshold_deg)
    )

    # Useful comparative scores
    df_ok["fitness_margin_ref_minus_empty"] = (
        df_ok["reference_final_fitness"] - df_ok["empty_final_fitness"]
    )
    df_ok["rmse_margin_empty_minus_ref"] = (
        df_ok["empty_final_rmse"] - df_ok["reference_final_rmse"]
    )
    df_ok["translation_error_margin_empty_minus_ref"] = (
        df_ok["empty_translation_error"] - df_ok["reference_translation_error"]
    )
    df_ok["rotation_error_margin_empty_minus_ref"] = (
        df_ok["empty_rotation_error_deg"] - df_ok["reference_rotation_error_deg"]
    )

    # Long format for easier aggregation/plotting
    df_long = pd.concat([
        pd.DataFrame({
            "variation_idx": df_ok["variation_idx"],
            "crop_idx": df_ok["crop_idx"],
            "target": "reference",
            "fraction_kept": df_ok["fraction_kept"],
            "num_kept": df_ok["num_kept"],
            "translation_error": df_ok["reference_translation_error"],
            "rotation_error_deg": df_ok["reference_rotation_error_deg"],
            "fitness": df_ok["reference_final_fitness"],
            "rmse": df_ok["reference_final_rmse"],
            "strict_success": df_ok["reference_strict_success"],
            "acceptable": df_ok["reference_acceptable"],
        }),
        pd.DataFrame({
            "variation_idx": df_ok["variation_idx"],
            "crop_idx": df_ok["crop_idx"],
            "target": "empty",
            "fraction_kept": df_ok["fraction_kept"],
            "num_kept": df_ok["num_kept"],
            "translation_error": df_ok["empty_translation_error"],
            "rotation_error_deg": df_ok["empty_rotation_error_deg"],
            "fitness": df_ok["empty_final_fitness"],
            "rmse": df_ok["empty_final_rmse"],
            "strict_success": df_ok["empty_strict_success"],
            "acceptable": df_ok["empty_acceptable"],
        }),
    ], ignore_index=True)

    summary_target = (
        df_long.groupby("target")
        .agg(
            trials=("target", "size"),
            mean_translation_error=("translation_error", "mean"),
            median_translation_error=("translation_error", "median"),
            mean_rotation_error_deg=("rotation_error_deg", "mean"),
            median_rotation_error_deg=("rotation_error_deg", "median"),
            mean_fitness=("fitness", "mean"),
            mean_rmse=("rmse", "mean"),
            strict_success_rate=("strict_success", "mean"),
            acceptable_rate=("acceptable", "mean"),
        )
        .reset_index()
    )

    summary_variation = (
        df_long.groupby(["variation_idx", "target"])
        .agg(
            trials=("target", "size"),
            mean_translation_error=("translation_error", "mean"),
            mean_rotation_error_deg=("rotation_error_deg", "mean"),
            mean_fitness=("fitness", "mean"),
            mean_rmse=("rmse", "mean"),
            strict_success_rate=("strict_success", "mean"),
            acceptable_rate=("acceptable", "mean"),
            mean_fraction_kept=("fraction_kept", "mean"),
            mean_num_kept=("num_kept", "mean"),
        )
        .reset_index()
        .sort_values(["variation_idx", "target"])
    )

    # Ranking view for quick inspection
    df_ranked = df_ok.copy()
    df_ranked["reference_badness_score"] = (
        df_ranked["reference_translation_error"] +
        0.1 * df_ranked["reference_rotation_error_deg"] -
        0.5 * df_ranked["reference_final_fitness"]
    )
    df_ranked["empty_badness_score"] = (
        df_ranked["empty_translation_error"] +
        0.1 * df_ranked["empty_rotation_error_deg"] -
        0.5 * df_ranked["empty_final_fitness"]
    )

    # --------------------
    # Plots
    # --------------------
    figs = {}

    # 1) Translation error by trial
    fig1 = plt.figure(figsize=(10, 4))
    x = np.arange(len(df_ok))
    plt.scatter(x, df_ok["reference_translation_error"], label="reference")
    plt.scatter(x, df_ok["empty_translation_error"], label="empty")
    plt.axhline(translation_success_threshold, linestyle="--", linewidth=1)
    plt.xlabel("trial index")
    plt.ylabel("translation error")
    plt.title("Translation error per trial")
    plt.legend()
    plt.tight_layout()
    figs["translation_error_per_trial"] = fig1

    # 2) Rotation error by trial
    fig2 = plt.figure(figsize=(10, 4))
    plt.scatter(x, df_ok["reference_rotation_error_deg"], label="reference")
    plt.scatter(x, df_ok["empty_rotation_error_deg"], label="empty")
    plt.axhline(rotation_success_threshold_deg, linestyle="--", linewidth=1)
    plt.xlabel("trial index")
    plt.ylabel("rotation error [deg]")
    plt.title("Rotation error per trial")
    plt.legend()
    plt.tight_layout()
    figs["rotation_error_per_trial"] = fig2

    # 3) Fitness comparison
    fig3 = plt.figure(figsize=(10, 4))
    plt.scatter(x, df_ok["reference_final_fitness"], label="reference")
    plt.scatter(x, df_ok["empty_final_fitness"], label="empty")
    plt.xlabel("trial index")
    plt.ylabel("fitness")
    plt.title("Fitness per trial")
    plt.legend()
    plt.tight_layout()
    figs["fitness_per_trial"] = fig3

    # 4) Fraction kept vs reference translation error
    fig4 = plt.figure(figsize=(6, 5))
    plt.scatter(df_ok["fraction_kept"], df_ok["reference_translation_error"])
    plt.xlabel("fraction kept")
    plt.ylabel("reference translation error")
    plt.title("Overlap proxy vs reference translation error")
    plt.tight_layout()
    figs["fraction_kept_vs_ref_translation_error"] = fig4

    # 5) Reference vs Empty fitness
    fig5 = plt.figure(figsize=(6, 6))
    plt.scatter(df_ok["empty_final_fitness"], df_ok["reference_final_fitness"])
    mn = min(df_ok["empty_final_fitness"].min(), df_ok["reference_final_fitness"].min())
    mx = max(df_ok["empty_final_fitness"].max(), df_ok["reference_final_fitness"].max())
    plt.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1)
    plt.xlabel("empty fitness")
    plt.ylabel("reference fitness")
    plt.title("Reference vs empty fitness")
    plt.tight_layout()
    figs["reference_vs_empty_fitness"] = fig5

    # 6) Success rate by variation
    success_by_variation = (
        df_ok.groupby("variation_idx")[["reference_strict_success", "empty_strict_success"]]
        .mean()
        .reset_index()
    )
    fig6 = plt.figure(figsize=(8, 4))
    width = 0.35
    xv = np.arange(len(success_by_variation))
    plt.bar(xv - width/2, success_by_variation["reference_strict_success"], width, label="reference")
    plt.bar(xv + width/2, success_by_variation["empty_strict_success"], width, label="empty")
    plt.xticks(xv, success_by_variation["variation_idx"])
    plt.ylim(0, 1)
    plt.xlabel("variation idx")
    plt.ylabel("strict success rate")
    plt.title("Strict success rate by variation")
    plt.legend()
    plt.tight_layout()
    figs["success_rate_by_variation"] = fig6

    # Print concise interpretation
    print("=== Per-target summary ===")
    print(summary_target.to_string(index=False))

    print("\n=== Per-variation summary ===")
    print(summary_variation.to_string(index=False))

    print("\n=== Best reference trials ===")
    best_ref = df_ok.sort_values(
        ["reference_translation_error", "reference_rotation_error_deg", "reference_final_rmse"]
    )[
        [
            "variation_idx", "crop_idx",
            "fraction_kept",
            "reference_translation_error", "reference_rotation_error_deg",
            "reference_final_fitness", "reference_final_rmse",
            "empty_translation_error", "empty_rotation_error_deg",
            "fitness_margin_ref_minus_empty",
        ]
    ]
    print(best_ref.head(5).to_string(index=False))

    print("\n=== Worst reference trials ===")
    worst_ref = df_ok.sort_values(
        ["reference_translation_error", "reference_rotation_error_deg"],
        ascending=False
    )[
        [
            "variation_idx", "crop_idx",
            "fraction_kept",
            "reference_translation_error", "reference_rotation_error_deg",
            "reference_final_fitness", "reference_final_rmse",
            "empty_translation_error", "empty_rotation_error_deg",
            "fitness_margin_ref_minus_empty",
        ]
    ]
    print(worst_ref.head(5).to_string(index=False))

    return df_ok, summary_target, summary_variation, figs

In [ ]:
df_results, summary_target, summary_variation, figs = summarize_partial_alignment_results(results)
display(df_results[[
    "variation_idx", "crop_idx", "fraction_kept",
    "reference_translation_error", "reference_rotation_error_deg",
    "reference_final_fitness", "reference_final_rmse",
    "empty_translation_error", "empty_rotation_error_deg",
    "empty_final_fitness", "empty_final_rmse",
    "fitness_margin_ref_minus_empty"
]].sort_values(["variation_idx", "crop_idx"]))

In [ ]:
def show_successful_alignments(
    results,
    ref_pcd,
    max_to_show=None,
    translation_error_thresh=0.10,
    rotation_error_thresh_deg=1.0,
    use_reference=True,
):
    """
    Display successful alignments from the results list using:

        draw_registration_result(
            ref_pcd,
            result["moved_pcd"],
            np.linalg.inv(result["estimated_transform"])
        ).show()

    Parameters
    ----------
    results : list[dict]
        Output from your evaluation function.

    ref_pcd : open3d.geometry.PointCloud
        Reference point cloud to visualize against.

    max_to_show : int or None
        Maximum number of successful results to display.

    translation_error_thresh : float
        Success threshold for translation error.

    rotation_error_thresh_deg : float
        Success threshold for rotation error.

    use_reference : bool
        If True uses reference_* metrics/transforms.
        If False uses empty_* metrics/transforms.
    """
    import numpy as np

    shown = 0

    if use_reference:
        prefix = "reference"
    else:
        prefix = "empty"
    scenes = []
    for result in results:
        if result.get("status") != "ok":
            continue

        t_err = result.get(f"{prefix}_translation_error", np.inf)
        r_err = result.get(f"{prefix}_rotation_error_deg", np.inf)

        if t_err <= translation_error_thresh and r_err <= rotation_error_thresh_deg:

            est_transform = result[f"{prefix}_estimated_transform"]

            print(
                f"variation={result['variation_idx']} "
                f"crop={result['crop_idx']} | "
                f"t_err={t_err:.4f} | "
                f"r_err={r_err:.4f}°"
            )

            scenes.append(
                draw_registration_result(
                    ref_pcd,
                    result["moved_pcd"],
                    np.linalg.inv(est_transform)
                )
            )

            shown += 1

            if max_to_show is not None and shown >= max_to_show:
                break

    print(f"Displayed {shown} successful alignments.")
    return scenes

In [ ]:
# Show all successful alignments against the reference scene
show_successful_alignments(
    results=results,
    ref_pcd=refPcd,
    use_reference = True
)[1].show()